# 00 — Colab/local setup and repo/NWB manifest

Resolves the NWB path, computes SHA256, creates the run output directory, and writes `manifests/run_manifest.json`.

In [1]:
import os
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _omission_run_common import (
    base_manifest,
    project_root,
    resolve_nwb_path,
    resolve_run_root,
    resolve_nwb_sha256,
    write_json,
    write_warnings,
)

start = time.time()
repo = project_root()
nwb_path = resolve_nwb_path(repo)
nwb_sha256 = resolve_nwb_sha256(nwb_path)
run_root = resolve_run_root(repo, nwb_path, nwb_sha256)
run_root.mkdir(parents=True, exist_ok=True)
os.environ["OMISSION_RUN_ROOT"] = str(run_root)

warnings = []
manifest = base_manifest(
    notebook_id="00_colab_setup_and_manifest",
    analysis_stage="setup",
    warnings_rel="warnings/00_warnings.json",
    runtime_seconds=time.time() - start,
    repo=repo,
    nwb_path=nwb_path,
    run_root=run_root,
    nwb_sha256=nwb_sha256,
    outputs=[
        "manifests/run_manifest.json",
        "reports/runtime_report.md",
        "warnings/00_warnings.json",
    ],
)

manifest_path = run_root / "manifests" / "run_manifest.json"
write_json(manifest_path, manifest)
write_warnings(run_root / "warnings" / "00_warnings.json", warnings)

report_md = run_root / "reports" / "runtime_report.md"
report_md.parent.mkdir(parents=True, exist_ok=True)
report_md.write_text(
    "# Notebook 00 — Setup\n\n"
    f"- Repo root: {manifest['repo_root']}\n"
    f"- Branch: {manifest['repo_branch']}\n"
    f"- Commit SHA: {manifest['repo_sha']}\n"
    f"- NWB source: {manifest['nwb_source_path']}\n"
    f"- NWB SHA256: {manifest['nwb_sha256']}\n"
    f"- Output directory: {manifest['output_root']}\n"
    f"- SMOKE_MODE: {manifest['smoke_mode']}\n",
    encoding="utf-8",
)

print("Setup complete.")
print("run_root:", run_root)
print("manifest:", manifest_path)

RuntimeError: NWB file not found via OMISSION_NWB_PATH, drive path, or data/*.nwb